# Figshare Document Pipeline

1. **Metadata** — batch-search Figshare for journal contributions matching DOIs in `dois.txt`; store `metadata.json` per article
2. **Details** — fetch full article details; store `details.json` per article
3. **Download** — download PDF/DOC/DOCX files into `raw_data/retrieved/{article_id}/`
4. **Conversion** — convert DOC/DOCX files to PDF using LibreOffice
5. **Text extraction** — extract plain text and bold blocks into `raw_data/extracted/{article_id}/`

In [ ]:
import json
import re
import tempfile
import time
from pathlib import Path

import fitz
import requests
import shutil
import subprocess
from tqdm.auto import tqdm

BASE_URL = "https://api.figshare.com/v2"
DOIS_FILE = Path("dois.txt")
assert DOIS_FILE.exists()

TRANSLATION_TABLE_FILE = Path("translation_table.json")
assert TRANSLATION_TABLE_FILE.exists()

RAW_DATA_DIR = Path("raw_data")
RETRIEVED_DIR = RAW_DATA_DIR / "retrieved"
EXTRACTED_DIR = RAW_DATA_DIR / "extracted"

DOI_TO_ARTICLE_ID_FILE = RAW_DATA_DIR / "doi_to_article_id.json"
BATCH_SIZE = 10

DOCUMENT_EXTENSIONS = (".pdf", ".docx", ".doc")
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds
TIMEOUT = 60  # seconds

for directory in [RETRIEVED_DIR, EXTRACTED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def atomic_json_write(data, dest):
    """Write *data* as JSON to *dest* atomically via a temp file + rename."""
    with tempfile.NamedTemporaryFile(
        "w", dir=dest.parent, delete=False, suffix=".tmp"
    ) as tmp:
        json.dump(data, tmp, indent=2)
    Path(tmp.name).replace(dest)

## 1. Load DOIs

In [ ]:
dois = [
    line.strip()
    for line in DOIS_FILE.read_text().splitlines()
    if line.strip() and not line.startswith("#")
]
print(f"Loaded {len(dois):,} DOIs from {DOIS_FILE}")

## 2. Batch search Figshare for metadata

In [ ]:
def _search_batch(doi_batch, page_size=100):
    """
    Search Figshare for articles whose related materials reference any DOI
    in *doi_batch*.  Returns the raw list of article search results.
    """
    query = " OR ".join(f":resource_doi: {d}" for d in doi_batch)
    data = {
        "search_for": query,
        "page_size": page_size,
        "order": "published_date",
        "order_direction": "desc",
    }
    resp = requests.post(f"{BASE_URL}/articles/search", json=data, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def retrieve_metadata(dois):
    n_batches = (len(dois) + BATCH_SIZE - 1) // BATCH_SIZE
    doi_to_article_id = {doi: [] for doi in dois}
    for i in tqdm(range(0, len(dois), BATCH_SIZE), total=n_batches, desc="Metadata"):
        batch = dois[i : i + BATCH_SIZE]

        try:
            articles = _search_batch(batch)
        except requests.RequestException as exc:
            tqdm.write(f"  ERROR — {exc}")
            time.sleep(2)
            continue

        for article in articles:
            if article["defined_type_name"] != "journal contribution":
                continue
            file_path = RETRIEVED_DIR / str(article["id"]) / "metadata.json"
            file_path.parent.mkdir(parents=True, exist_ok=True)
            atomic_json_write(article, file_path)
            doi = article["resource_doi"]
            doi_to_article_id[doi].append(article["id"])

        time.sleep(1)

    return doi_to_article_id

In [ ]:
with open(DOI_TO_ARTICLE_ID_FILE) as f:
    doi_to_article_id = json.load(f)

unseen_dois = [
    doi
    for doi in dois
    if doi not in doi_to_article_id or not all(
        (RETRIEVED_DIR / str(article_id) / "metadata.json").exists()
        for article_id in doi_to_article_id[doi]
    )
]
print(f"Found {len(unseen_dois):,} unseen DOIs")

new_doi_to_article_id = retrieve_metadata(unseen_dois)

print(
    f"Searched {len(unseen_dois):,} new DOIs, {len(new_doi_to_article_id):,} found"
)

doi_to_article_id.update(new_doi_to_article_id)
atomic_json_write(doi_to_article_id, DOI_TO_ARTICLE_ID_FILE)

print(f"Metadata files: {len(list(RETRIEVED_DIR.glob('**/metadata.json'))):,}")

## 3. Retrieve article details

In [ ]:
def fetch_article_detail(article_id):
    resp = requests.get(f"{BASE_URL}/articles/{article_id}", timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def retrieve_details(article_ids):
    for id in tqdm(article_ids, desc="Details"):
        dest = RETRIEVED_DIR / str(id) / "details.json"
        if dest.exists():
            continue

        for attempt in range(MAX_RETRIES):
            try:
                detail = fetch_article_detail(id)
                atomic_json_write(detail, dest)
                break
            except requests.RequestException as exc:
                tqdm.write(f"  {id} attempt {attempt + 1} — {exc}")
                time.sleep(2 ** (attempt + 1))

        time.sleep(1)

In [ ]:
article_ids = [
    path.stem
    for path in RETRIEVED_DIR.glob("*")
    if path.is_dir() and path.name.isdigit()
]
unseen_ids = [
    id for id in article_ids
    if not (RETRIEVED_DIR / str(id) / "details.json").exists()
]

retrieve_details(sorted(unseen_ids))
print(f"Detail files: {len(list(RETRIEVED_DIR.glob('**/details.json'))):,}")

## 4. Download documents from Figshare

In [ ]:
missing = []
already_downloaded = 0

for detail_file in sorted(RETRIEVED_DIR.glob("*/details.json")):
    with open(detail_file) as f:
        article = json.load(f)

    article_id = detail_file.parent.name

    for file_entry in article.get("files", []):
        filename = file_entry["name"]
        if not filename.lower().endswith(DOCUMENT_EXTENSIONS):
            continue
        if (RETRIEVED_DIR / article_id / filename).exists():
            already_downloaded += 1
            continue
        missing.append(
            {
                "filename": filename,
                "download_url": file_entry["download_url"],
                "article_id": article_id,
            }
        )

print(f"Documents already downloaded: {already_downloaded}")
print(f"Documents to download:        {len(missing):,}")

In [ ]:
downloaded = 0
failed = []

session = requests.Session()

for entry in tqdm(missing, desc="Downloading"):
    dest_dir = RETRIEVED_DIR / str(entry["article_id"])
    dest = dest_dir / entry["filename"]
    if dest.exists():
        downloaded += 1
        continue

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(entry["download_url"], timeout=TIMEOUT)
            resp.raise_for_status()
            with tempfile.NamedTemporaryFile(
                dir=dest_dir, delete=False, suffix=".tmp"
            ) as tmp:
                tmp.write(resp.content)
                tmp_path = Path(tmp.name)
            tmp_path.rename(dest)

            downloaded += 1
            break
        
        except Exception as exc:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
            else:
                failed.append({**entry, "error": str(exc)})

print(f"\nDownloaded: {downloaded}")
print(f"Failed:     {len(failed)}")

In [ ]:
if failed:
    print("Failed downloads:")
    for entry in failed:
        print(
            f"  {entry['filename']} (article {entry['article_id']}): {entry['error']}"
        )

## 5. Convert DOC/DOCX documents to PDF

In [ ]:
libreoffice_available = shutil.which("libreoffice") is not None

converted_from_word = 0

if libreoffice_available:
    doc_files = sorted(
        p for p in RETRIEVED_DIR.glob("*/*")
        if p.suffix.lower() in (".doc", ".docx")
    )
    print(f"Documents to convert: {len(doc_files)}")

    conv_failed = []

    for src in tqdm(doc_files, desc="Converting"):
        pdf_dest = src.with_suffix(".pdf")
        if pdf_dest.exists():
            converted_from_word += 1
            continue

        try:
            with tempfile.TemporaryDirectory(dir=src.parent) as tmp_dir:
                result = subprocess.run(
                    [
                        "libreoffice",
                        "--headless",
                        "--convert-to",
                        "pdf",
                        "--outdir",
                        tmp_dir,
                        str(src),
                    ],
                    capture_output=True,
                    text=True,
                    timeout=120,
                )
                tmp_pdf = Path(tmp_dir) / (src.stem + ".pdf")
                if result.returncode == 0 and tmp_pdf.exists():
                    tmp_pdf.rename(pdf_dest)
                    src.unlink()
                    converted_from_word += 1
                else:
                    conv_failed.append((src.name, result.stderr.strip()))
        except Exception as exc:
            conv_failed.append((src.name, str(exc)))

    print(f"Converted: {converted_from_word}")
    print(f"Failed:    {len(conv_failed)}")
    if conv_failed:
        for name, err in conv_failed:
            print(f"  {name}: {err}")

else:
    print("libreoffice not found in PATH; install it to convert .doc/.docx files")

## 6. Extract text and locate bold text blocks

In [ ]:
BOLD_FLAG = 1 << 4
COMMON_FRAG_PATTERN = re.compile(r"(yl|ox)")
STEREO_PATTERN = re.compile(
    r"""
    \(
        (?:\d+[RS]|[EZ])               # first descriptor
        (?:\s*,\s*(?:\d+[RS]|[EZ]))*   # optional additional ones
    \)
    """,
    re.VERBOSE,
)
with open("translation_table.json") as f:
    TRANSLATION_TABLE = str.maketrans(json.load(f))


class BoldBlock(dict):
    def __init__(self, start, stop, full_text):
        super().__init__()
        self["type"] = "bold"
        self["start"] = start
        self["stop"] = stop
        self["text"] = full_text[start:stop]


def has_stereo(text):
    return bool(STEREO_PATTERN.search(text))


def has_common_fragment(text):
    return bool(COMMON_FRAG_PATTERN.search(text))


def is_worth_keeping(text):
    return has_stereo(text) or has_common_fragment(text)


def _is_bold(span):
    return span["flags"] & BOLD_FLAG or "bold" in span["font"].lower()


def _is_symbols(text):
    return all(not (c.isalnum() or c in " \n\f") for c in text)


def _add_text(text, full_text, pos):
    return full_text + text, pos + len(text)


def _add_char_if_needed(char, full_text, pos):
    if full_text and full_text[-1] != char:
        return full_text + char, pos + 1
    return full_text, pos


def extract_text(pdf):
    full_text = ""
    pos = 0

    blocks = []
    bold_text = ""
    bold_start = bold_stop = None

    with fitz.open(str(pdf)) as doc:
        for page in doc:
            for block in page.get_text("dict")["blocks"]:
                for line in block.get("lines", []):
                    for span in line["spans"]:
                        text = span["text"].translate(TRANSLATION_TABLE)
                        if not text.isprintable():
                            continue
                        if _is_bold(span) or (bold_text and _is_symbols(text)):
                            if bold_text == "":
                                bold_start = pos
                            bold_text += text
                            bold_stop = pos + len(text)
                        elif bold_text != "":
                            if is_worth_keeping(bold_text):
                                blocks.append(
                                    BoldBlock(bold_start, bold_stop, full_text)
                                )
                            bold_text = ""
                        full_text, pos = _add_text(text, full_text, pos)
                    full_text, pos = _add_char_if_needed(" ", full_text, pos)
                full_text, pos = _add_char_if_needed(" ", full_text, pos)
            full_text, pos = _add_char_if_needed("\n", full_text, pos)

    if bold_text and is_worth_keeping(bold_text):
        blocks.append(BoldBlock(bold_start, bold_stop, full_text))

    return full_text, blocks

In [ ]:
pdf_files = sorted(RETRIEVED_DIR.glob("*/*.pdf"))
print(f"Total PDF files: {len(pdf_files)}")

remaining_files = []
for pdf in pdf_files:
    article_id = pdf.parent.name
    stem = f"{article_id}/{pdf.stem}"
    article_dir = EXTRACTED_DIR / article_id
    text_file = article_dir / f"{pdf.stem}.txt"
    bold_file = article_dir / f"{pdf.stem}.json"
    if not (text_file.exists()):
        remaining_files.append((pdf, article_dir, stem, text_file, bold_file))
print(f"Remaining PDF files to extract text from: {len(remaining_files)}")

for pdf, article_dir, stem, text_file, bold_file in tqdm(remaining_files, desc="Extracting"):
    article_dir.mkdir(parents=True, exist_ok=True)
    pdf_text, bold_blocks = extract_text(pdf)
    text_file.write_text(pdf_text)
    atomic_json_write(bold_blocks, bold_file)

## 7. Summary

In [ ]:
print(f"DOIs loaded:    {len(dois):,}")
print(f"Metadata files: {len(list(RETRIEVED_DIR.glob('*/metadata.json'))):,}")
print(f"Detail files:   {len(list(RETRIEVED_DIR.glob('*/details.json'))):,}")
print(f"PDF Documents:  {len(pdf_files)}")
print(f"Extracted text: {len(list(EXTRACTED_DIR.glob('*/*.txt'))):,}")